# Conditional model experiments
Загружаем лучшую EMA-модель, берём 18 равномерно расставленных сэмплов из val,
применяем маску треков и смотрим качество восстановления.

In [ ]:
import os, sys, json
import numpy as np
import torch
import matplotlib.pyplot as plt
from functools import partial

sys.path.insert(0, os.path.dirname(os.getcwd()) if 'diffusion_data_assimilation' not in os.getcwd() else os.getcwd())

from diffusers.models.unets.unet_2d import UNet2DModel
from diffusers.training_utils import EMAModel

from utils import (
    NpyImageDataset, channel_normalize,
    generate_satellite_track_mask, get_device,
    make_plot, make_difference_plot,
)
from sampler import Sampler

## Конфигурация

In [ ]:
# ── пути ──────────────────────────────────────────────────────────────────────
CHECKPOINT_DIR = "checkpoints"          # папка с run_*
RUN_NAME       = None                   # None → возьмёт последний run
DATA_ROOT      = "/mnt/sciml/a.sadreev/sea_ice_data"  # сервер
# DATA_ROOT    = "/Users/amir/sciml/sea_ice_data"     # mac

# ── параметры эксперимента ────────────────────────────────────────────────────
N_SAMPLES      = 18          # make_plot / make_difference_plot рисуют 3×6=18
NUM_TIMESTEPS  = 50
N_TRACKS_RANGE = (1, 4)
IMAGE_SIZE     = (320, 256)

DEVICE = get_device()
print(f"device: {DEVICE}")

# ── нормализация (из stats.json) ──────────────────────────────────────────────
with open(os.path.join(DATA_ROOT, "train", "stats.json")) as f:
    stats = json.load(f)
CHANNEL_MEAN = tuple(stats["mean"])
CHANNEL_STD  = tuple(stats["std"])
print(f"channel_mean={CHANNEL_MEAN}  channel_std={CHANNEL_STD}")

## Загрузка модели

In [ ]:
# Найти последний (или указанный) run
if RUN_NAME is None:
    runs = sorted(d for d in os.listdir(CHECKPOINT_DIR) if d.startswith("run_"))
    assert runs, f"Нет run_* в {CHECKPOINT_DIR}"
    RUN_NAME = runs[-1]
run_dir = os.path.join(CHECKPOINT_DIR, RUN_NAME)
print(f"Используем run: {run_dir}")

# Если есть config.json в run — распечатаем ключевые параметры
cfg_path = os.path.join(run_dir, "config.json")
if os.path.exists(cfg_path):
    with open(cfg_path) as f:
        saved_cfg = json.load(f)
    print(json.dumps({k: v for k, v in saved_cfg.items()
                      if k not in ("channel_mean", "channel_std")}, indent=2))

In [ ]:
model = UNet2DModel(
    sample_size=IMAGE_SIZE,
    in_channels=7,
    out_channels=2,
    layers_per_block=2,
    block_out_channels=(64, 128, 256, 512, 512),
    down_block_types=("DownBlock2D", "DownBlock2D", "DownBlock2D",
                      "AttnDownBlock2D", "DownBlock2D"),
    up_block_types=("UpBlock2D", "AttnUpBlock2D", "UpBlock2D",
                    "UpBlock2D", "UpBlock2D"),
)

ema_path = os.path.join(run_dir, "ema_best_model.pth")
assert os.path.exists(ema_path), f"Файл не найден: {ema_path}"

ema = EMAModel(model.parameters(), decay=0.999)
ema.load_state_dict(torch.load(ema_path, map_location="cpu"))
ema.copy_to(model.parameters())

model.eval().to(DEVICE)
sampler = Sampler(model)
print("Модель загружена.")

## Валидационные данные — 18 равномерно расставленных сэмплов

In [ ]:
transform = partial(channel_normalize, channel_mean=CHANNEL_MEAN, channel_std=CHANNEL_STD)

val_dataset = NpyImageDataset(
    folder=os.path.join(DATA_ROOT, "valid"),
    transform=transform,
    preload=False,
    mmap_mode='r',
)
print(f"Всего val-сэмплов: {len(val_dataset)}")

# Равномерно выбираем N_SAMPLES индексов
indices = np.linspace(0, len(val_dataset) - 1, N_SAMPLES, dtype=int)
print(f"Выбраны индексы: {indices}")

clean_images = torch.stack([val_dataset[i] for i in indices]).to(DEVICE)  # (12, 2, H, W)
print(f"clean_images shape: {clean_images.shape}")

## Генерация маски трека

In [ ]:
valid_mask_npy = np.load(os.path.join(DATA_ROOT, "mask_padding.npy")).astype(np.float32)

mask_np = generate_satellite_track_mask(
    image_size=IMAGE_SIZE,
    batch_size=1,
    valid_mask=valid_mask_npy,
    n_tracks_range=N_TRACKS_RANGE,
)  # (1, H, W)

mask = torch.from_numpy(mask_np).unsqueeze(1).to(DEVICE)  # (1, 1, H, W)

fig, ax = plt.subplots(figsize=(5, 6))
ax.imshow(mask_np[0], cmap='gray', vmin=0, vmax=1)
ax.set_title("Satellite track mask")
ax.axis('off')
plt.tight_layout()
plt.show()

## Inference — обусловленный сэмплинг для каждого из 18 сэмплов

In [ ]:
predictions = []  # список (1, 2, H, W)

for i in range(N_SAMPLES):
    clean = clean_images[i:i+1]       # (1, 2, H, W)
    observed = clean * mask            # (1, 2, H, W)

    pred = sampler.sample_conditioned(
        mask=mask,
        observed=observed,
        size=IMAGE_SIZE,
        num_timesteps=NUM_TIMESTEPS,
        device=DEVICE,
    )  # (1, 2, H, W)

    predictions.append(pred.cpu())
    print(f"  {i+1}/{N_SAMPLES} done")

predictions = torch.cat(predictions, dim=0)  # (12, 2, H, W)
print(f"predictions shape: {predictions.shape}")

## Визуализация

In [ ]:
clean_cpu = clean_images.cpu()
pred_cpu  = predictions.cpu()

# Наблюдения: обнуляем всё вне треков
observed_cpu = clean_cpu * mask[0].cpu()  # (N, 2, H, W)

make_plot(clean_cpu,    CHANNEL_MEAN, CHANNEL_STD, N_SAMPLES, title="Truth — Concentration (channel 0)")
make_plot(observed_cpu, CHANNEL_MEAN, CHANNEL_STD, N_SAMPLES, title="Observed on tracks — Concentration (channel 0)")
make_plot(pred_cpu,     CHANNEL_MEAN, CHANNEL_STD, N_SAMPLES, title="Prediction — Concentration (channel 0)")

In [ ]:
make_difference_plot(clean_cpu, pred_cpu, CHANNEL_MEAN, CHANNEL_STD, N_SAMPLES)

## Средний MSE по треку (на валидационных сэмплах)

In [ ]:
truth_dn = channel_denormalize(clean_images.clone().cpu(), CHANNEL_MEAN, CHANNEL_STD)
pred_dn  = channel_denormalize(predictions.clone().cpu(), CHANNEL_MEAN, CHANNEL_STD)
mask_cpu = mask[0, 0].cpu()  # (H, W)

diff2 = (truth_dn - pred_dn) ** 2  # (12, 2, H, W)
mask_exp = mask_cpu.unsqueeze(0).unsqueeze(0)  # (1, 1, H, W)

n_pixels = mask_cpu.sum().item()
mse_per_sample = (diff2 * mask_exp).sum(dim=(2, 3)) / n_pixels  # (12, 2)

ch_names = ["Concentration", "Thickness"]
print(f"{'Sample':>8}  {'Concentration MSE':>18}  {'Thickness MSE':>14}")
print("-" * 46)
for i in range(N_SAMPLES):
    print(f"  #{indices[i]:>5}  {mse_per_sample[i,0].item():>18.5f}  {mse_per_sample[i,1].item():>14.5f}")
print("-" * 46)
print(f"  {'mean':>6}  {mse_per_sample[:,0].mean().item():>18.5f}  {mse_per_sample[:,1].mean().item():>14.5f}")